# Trening 5 algorytmow detekcji anomalii (Colab)

Notebook trenuje warstwe 3 detektora (`detection/ml.py`) na **Drone Telemetry Tampering Dataset v2**
uzywajac 5 algorytmow i porownuje skutecznosc.

## Co musisz miec przygotowane lokalnie

1. **`drone_anomaly.zip`** - spakowany projekt. Polecenie do pakowania (uruchom w WSL w katalogu projektu):

   ```bash
   zip -r drone_anomaly.zip . \
       -x "*.git*" "*__pycache__*" "data/*.csv" "models/*" "*.ipynb_checkpoints*"
   ```

2. **`kaggle.json`** - token API z Kaggle (Account -> Settings -> API -> Create New Token).

Oba pliki bedziesz uploadowac komorkami ponizej.

## Wynik
5 pickle'i modeli + tabele porownawcze + wykresy ROC, spakowane w `results.zip` do pobrania na koncu.

## 1. Instalacja pakietow

In [ ]:
!pip install -q kaggle xgboost

## 2. Upload kodu projektu (drone_anomaly.zip)

Po wybraniu pliku ZIP w dialogu - zostanie rozpakowany do `/content/drone_anomaly` i dodany do `sys.path`.

In [ ]:
from google.colab import files
import os, sys, shutil

PROJECT_DIR = '/content/drone_anomaly'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

uploaded = files.upload()  # wybierz drone_anomaly.zip
zip_name = next(iter(uploaded))

!unzip -q {zip_name} -d {PROJECT_DIR}
!ls {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

## 3. Upload tokenu Kaggle (kaggle.json) i pobranie datasetu

In [ ]:
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()  # wybierz kaggle.json
    !mkdir -p /root/.kaggle
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

!mkdir -p data models
!kaggle datasets download -d rasikaekanayakadevlk/drone-telemetry-tampering-dataset-v2 -p data --unzip
!ls -lh data/

In [ ]:
# Loader szuka domyslnie pliku `data/drone_telemetry_v2.csv`.
# Jesli po unzip nazwa jest inna - znajdz ja i przemianuj:
import glob
csvs = glob.glob('data/*.csv')
print('Znalezione CSV:', csvs)

TARGET = 'data/drone_telemetry_v2.csv'
if csvs and not os.path.exists(TARGET):
    os.rename(csvs[0], TARGET)
    print(f'Przemianowano {csvs[0]} -> {TARGET}')

## 4. Wczytanie i podsumowanie datasetu

In [ ]:
from data.loader import load_dataset, split_by_replicate, summarize

df = load_dataset('data/drone_telemetry_v2.csv')
summarize(df)

train, test = split_by_replicate(df)
print(f'\nTrain (replicate 0,1,2): {len(train):,}')
print(f'Test  (replicate 3):     {len(test):,}')

## 5. Warstwy 1 + 2 na zbiorze testowym (raz, nie zaleza od algorytmu W3)

In [ ]:
from detection.rules import detect_threshold_violations
from detection.statistical import detect_sudden_changes

test_base = detect_threshold_violations(test)
test_base = detect_sudden_changes(test_base)
print(f'Alerty W1: {test_base["alert_threshold"].sum():,}')
print(f'Alerty W2: {test_base["alert_change"].sum():,}')

## 6. Cechy okienkowe - liczone RAZ dla train/test

`compute_features` na milionach wierszy trwa kilka minut. Liczymy je tutaj raz
i przekazujemy do `fit/predict` w petli (zamiast 10 wywolan rolling/groupby -
tylko 2). Oszczednosc ~60 min na pelnym datasecie.

In [ ]:
import time
from detection.ml import compute_features

t0 = time.time()
train_features = compute_features(train, window=8)
print(f'train_features: {train_features.shape}  ({time.time() - t0:.1f}s)')

t0 = time.time()
test_features = compute_features(test_base, window=8)
print(f'test_features:  {test_features.shape}  ({time.time() - t0:.1f}s)')

## 7. Trening 5 algorytmow warstwy 3 + zapis modeli

Dla OCSVM i LOF `max_train_samples=50_000` (default w `AnomalyDetector`) - bez tego
trening O(n^2) by sie nigdy nie skonczyl na milionach probek. Mozna nadpisac
per algorytm, np. `AnomalyDetector('lof', max_train_samples=20_000)`.

In [ ]:
import pandas as pd

from detection.ml import AnomalyDetector
from evaluation.metrics import compute_layer_metrics

ALGORITHMS = ['isolation_forest', 'one_class_svm', 'lof', 'random_forest', 'xgboost']

results = {}
predictions = {}

for algo in ALGORITHMS:
    print(f'\n=== {algo} ===')
    t0 = time.time()
    try:
        det = AnomalyDetector(algorithm=algo).fit(train, features=train_features)
        out = det.predict(test_base, features=test_features)
        det.save(f'models/{algo}.pkl')
        m = compute_layer_metrics(out, 'alert_ml')
        results[algo] = {**m, 'time_s': round(time.time() - t0, 1)}
        predictions[algo] = out
        print(f'  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  '
              f'(czas: {results[algo]["time_s"]}s)')
    except Exception as e:
        print(f'  BLAD: {e}')
        results[algo] = None

summary = pd.DataFrame({k: v for k, v in results.items() if v}).T
print('\nPodsumowanie:')
summary

## 8. Krzywe ROC - porownanie wszystkich algorytmow na jednym wykresie

In [ ]:
from evaluation.metrics import plot_roc_curves

merged = test_base.copy()
score_columns = {}
UNSUP = {'isolation_forest', 'one_class_svm', 'lof'}
for algo, pred in predictions.items():
    col = f'score_{algo}'
    merged[col] = pred['ml_score'].values
    direction = 'low' if algo in UNSUP else 'high'
    score_columns[algo] = (col, direction)

plot_roc_curves(
    merged,
    score_columns,
    binary_layers={'alert_threshold': 'W1', 'alert_change': 'W2'},
    output_path='data/roc_curves_all_algos.png',
)

## 9. Pelna ewaluacja dla najlepszego algorytmu (przykladowo IF)

In [ ]:
from evaluation.metrics import run_full_evaluation

best = predictions['isolation_forest']  # albo dowolny inny
run_full_evaluation(
    best,
    score_columns={'isolation_forest': ('ml_score', 'low')},
    output_dir='data',
)

## 10. Spakuj wyniki i pobierz lokalnie

In [ ]:
!zip -r results.zip models/ data/*.png data/*.csv
files.download('results.zip')